# Fish Audio S2 Pro Colab Experiment

API-first Colab workflow for Fish Audio S2 Pro TTS / authorized voice cloning.

- The notebook clones `https://github.com/fishaudio/fish-speech` directly and runs Fish Speech from that checkout.
- It defaults to one long-lived authenticated API server for speed; Gradio is off because it loads a second model engine.
- It does not mount Google Drive by default. Outputs/logs stay under ephemeral `/content/fishaudio_s2_pro/artifacts` and can be downloaded from a cell.
- HF model files stay in ephemeral `/content`; do not store model artifacts in Drive.
- Use only voices/audio you are authorized to clone or synthesize.


In [ ]:
#@title 0. Runtime parameters
FISH_SPEECH_REPO_URL = "https://github.com/fishaudio/fish-speech.git"  #@param {type:"string"}
FISH_SPEECH_REF = "02995ed7bd61ea383727a2c173a3ce126965219c"  #@param {type:"string"}
FISH_REPO_PATH = "/content/fish-speech"  #@param {type:"string"}
WORK_ROOT_PATH = "/content/fishaudio_s2_pro"  #@param {type:"string"}
ARTIFACT_ROOT_PATH = "/content/fishaudio_s2_pro/artifacts"  #@param {type:"string"}
RUN_ID = ""  #@param {type:"string"}
MOUNT_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT_PATH = "/content/drive/MyDrive/voice/fishaudio-s2-pro"  # kept only for optional Drive mode
DRY_RUN = False  #@param {type:"boolean"}
APPLY_FAST_SOURCE_PATCHES = True  #@param {type:"boolean"}

FISH_UV_EXTRA_REQUEST = "auto"  #@param ["auto", "cu126", "cu128", "cu129"]
HF_DOWNLOAD_WORKERS = 2  #@param {type:"integer"}
API_PORT = 8080  #@param {type:"integer"}
ENABLE_API_AUTH = True  #@param {type:"boolean"}
API_AUTH_TOKEN = ""  #@param {type:"string"}
API_AUTH_SECRET_NAME = "FISHAUDIO_API_KEY"  #@param {type:"string"}
COMPILE_MODEL = False  #@param {type:"boolean"}
USE_HALF = False  #@param {type:"boolean"}

SHOWCASE_TEXT = "[laugh]或者开启代理中转策略  分享Pikpak：有大小限制，超出指定文件大小后只能播放40%~50%"  #@param {type:"string"}
SHOWCASE_MAX_NEW_TOKENS = 384  #@param {type:"integer"}
SHOWCASE_CHUNK_LENGTH = 1000  #@param {type:"integer"}
SHOWCASE_FORMAT = "wav"  #@param ["wav", "mp3", "opus"]
SHOWCASE_STREAMING = False  #@param {type:"boolean"}
DOWNLOAD_SHOWCASE_OUTPUT = False  #@param {type:"boolean"}

UPLOAD_REFERENCE_AUDIO = False  #@param {type:"boolean"}
REFERENCE_AUDIO_PATH = ""  #@param {type:"string"}
REFERENCE_TEXT = ""  #@param {type:"string"}
VOICE_ID = "demo_voice"  #@param {type:"string"}
REFERENCE_START_SECONDS = 0.0  #@param {type:"number"}
REFERENCE_MAX_SECONDS = 10.0  #@param {type:"number"}
SAVE_REFERENCE_COPY_TO_ARTIFACTS = False  #@param {type:"boolean"}
REFERENCE_TTS_TEXT = "This is a short reference voice smoke test."  #@param {type:"string"}

BUILD_AWESOME_WEBUI = False  #@param {type:"boolean"}
START_GRADIO_WEBUI = False  #@param {type:"boolean"}
PUBLIC_TUNNEL_MODE = "none"  #@param ["none", "cloudflare_quick", "cloudflare_named"]
CLOUDFLARED_TUNNEL_TOKEN_SECRET = "CLOUDFLARED_TUNNEL_TOKEN"  #@param {type:"string"}
CLOUDFLARED_CUSTOM_DOMAIN = ""  #@param {type:"string"}
DOWNLOAD_RUN_ZIP = False  #@param {type:"boolean"}


In [ ]:
#@title 1. Runtime helpers and environment checks
from pathlib import Path
import json, os, re, secrets, shlex, shutil, signal, subprocess, sys, time, urllib.request, wave

FISH_REPO = Path(FISH_REPO_PATH)
WORK_ROOT = Path(WORK_ROOT_PATH)
if MOUNT_DRIVE:
    ARTIFACT_ROOT = Path(DRIVE_ROOT_PATH)
else:
    ARTIFACT_ROOT = Path(ARTIFACT_ROOT_PATH)
DRIVE_ROOT = ARTIFACT_ROOT  # backward-compatible alias for older cells/scripts
DRY_RUN = bool(DRY_RUN)
source_patches = []
public_api_base_url = None
awesome_web_url = None

if DRY_RUN:
    os.environ.setdefault("FISH_NOTEBOOK_VALIDATE", "1")

if sys.version_info[:2] != (3, 12) and not DRY_RUN and not os.environ.get("FISH_NOTEBOOK_VALIDATE"):
    raise RuntimeError(f"This experiment targets Python 3.12; current kernel is {sys.version}")

def run(cmd, *, cwd=None, env=None, check=True):
    started = time.time()
    display_cwd = str(cwd) if cwd else os.getcwd()
    print(f"\n[{time.strftime('%H:%M:%S')}] cwd={display_cwd}", flush=True)
    print("$", " ".join(map(str, cmd)), flush=True)
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc_env["PYTHONUNBUFFERED"] = "1"
    proc_env.setdefault("PIP_PROGRESS_BAR", "on")
    proc_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    result = subprocess.run(cmd, cwd=cwd, env=proc_env, check=check)
    print(f"[{time.strftime('%H:%M:%S')}] done in {time.time() - started:.1f}s", flush=True)
    return result

def default_run_id(prefix="fishaudio_s2_pro"):
    return f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}"

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path

def write_silent_wav(path, *, sample_rate=16000, seconds=0.25):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = bytes(max(2, int(sample_rate * seconds) * 2))
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)
        wav.writeframes(frames)
    return path

def read_tail(path, lines=80):
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])

def require_vars(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError("Notebook state is missing required values: " + ", ".join(missing) + ". Re-run the earlier numbered cells in order; do not upload a new notebook.")

def read_colab_secret(name, *, required=True):
    value = os.environ.get(name)
    if value:
        return value
    if DRY_RUN:
        return f"dry-run-{name.lower()}" if required else None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Could not read Colab Secret {name}: {exc!r}") from exc
        return None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

def gpu_inventory():
    if DRY_RUN:
        return [{"name": "dry-run-gpu", "memory_mb": 81920}]
    result = subprocess.run([
        "nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits",
    ], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    gpus = []
    for line in result.stdout.splitlines():
        if not line.strip() or "," not in line:
            continue
        name, memory_mb = [part.strip() for part in line.split(",", 1)]
        gpus.append({"name": name, "memory_mb": int(memory_mb)})
    return gpus

def require_minimum_vram(minimum_gb=24):
    gpus = gpu_inventory()
    if not gpus:
        raise RuntimeError("No NVIDIA GPU detected. Use a Colab A100/H100 runtime.")
    best_mb = max(int(gpu["memory_mb"]) for gpu in gpus)
    if best_mb < minimum_gb * 1024:
        raise RuntimeError(f"Fish Audio S2 Pro needs at least {minimum_gb} GB VRAM; largest detected GPU has {best_mb / 1024:.1f} GB.")
    return gpus

def resolve_fish_uv_extra(value="auto"):
    if value != "auto":
        if value not in {"cu126", "cu128", "cu129", "cpu"}:
            raise ValueError("FISH_UV_EXTRA must be auto, cu126, cu128, cu129, or cpu")
        return value
    if DRY_RUN:
        return "cu128"
    result = subprocess.run(["nvidia-smi"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    match = re.search(r"CUDA Version:\s*([0-9]+)\.([0-9]+)", result.stdout or "")
    if not match:
        return "cu126"
    version = (int(match.group(1)), int(match.group(2)))
    if version >= (12, 9):
        return "cu129"
    if version >= (12, 8):
        return "cu128"
    return "cu126"

def api_headers(extra=None):
    headers = dict(extra or {})
    if globals().get("API_AUTH_TOKEN"):
        headers["Authorization"] = f"Bearer {API_AUTH_TOKEN}"
    return headers

def wait_for_http(url, *, headers=None, timeout_seconds=600, interval_seconds=2.0):
    if DRY_RUN:
        print("Dry run: skipping HTTP wait for", url)
        return True
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=10) as response:
                if 200 <= response.status < 500:
                    return True
        except Exception as exc:
            last_error = exc
            time.sleep(interval_seconds)
    raise RuntimeError(f"Timed out waiting for {url}: {last_error!r}")

def start_process(name, command, *, cwd, log_path, env=None, display_command=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    if DRY_RUN:
        log_path.write_text(f"dry run: skipped starting {name}\n", encoding="utf-8")
        info = {"name": name, "pid": 0, "command": display_command or command, "log_path": str(log_path), "dry_run": True}
        print(f"Dry run: skipped starting {name}; log={log_path}", flush=True)
        return info
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc_env.setdefault("PYTHONUNBUFFERED", "1")
    with log_path.open("ab") as log_file:
        proc = subprocess.Popen(
            command,
            cwd=Path(cwd),
            env=proc_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    info = {"name": name, "pid": proc.pid, "command": display_command or command, "log_path": str(log_path)}
    print(f"Started {name}: pid={proc.pid} log={log_path}", flush=True)
    return info

def stop_process(proc_info, *, timeout_seconds=30):
    if not proc_info or proc_info.get("dry_run"):
        return
    pid = int(proc_info["pid"])
    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            os.killpg(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.5)
    try:
        os.killpg(pid, signal.SIGKILL)
    except ProcessLookupError:
        return

def fish_env():
    env = os.environ.copy()
    env.update({
        "HF_HOME": str(CACHE_ROOT / "hf"),
        "HF_HUB_CACHE": str(CACHE_ROOT / "hf" / "hub"),
        "TRANSFORMERS_CACHE": str(CACHE_ROOT / "hf" / "transformers"),
        "PYTHONUNBUFFERED": "1",
        "FISH_CLEAR_CUDA_CACHE_AFTER_REQUEST": "0",
    })
    return env

def start_api_server(*, port=8080, compile_model=False, half=False, api_key=None):
    command = [
        "uv", "run", "python", "tools/api_server.py",
        "--listen", f"127.0.0.1:{port}",
        "--llama-checkpoint-path", str(CHECKPOINT_DIR),
        "--decoder-checkpoint-path", str(CHECKPOINT_DIR / "codec.pth"),
        "--decoder-config-name", "modded_dac_vq",
        "--device", "cuda",
        "--workers", "1",
    ]
    if compile_model:
        command.append("--compile")
    if half:
        command.append("--half")
    if api_key:
        command.extend(["--api-key", api_key])
    display = ["REDACTED" if api_key and part == api_key else part for part in command]
    return start_process("fish-api", command, cwd=FISH_REPO, log_path=LOGS_DIR / f"fish_api_{port}.log", env=fish_env(), display_command=display)

def start_gradio_webui(*, compile_model=False, half=False, theme="light"):
    command = [
        "uv", "run", "python", "tools/run_webui.py",
        "--llama-checkpoint-path", str(CHECKPOINT_DIR),
        "--decoder-checkpoint-path", str(CHECKPOINT_DIR / "codec.pth"),
        "--decoder-config-name", "modded_dac_vq",
        "--device", "cuda",
        "--theme", theme,
    ]
    if compile_model:
        command.append("--compile")
    if half:
        command.append("--half")
    return start_process("fish-gradio", command, cwd=FISH_REPO, log_path=LOGS_DIR / "fish_gradio.log", env=fish_env())

def patch_text_file(path, replacements):
    path = Path(path)
    if not path.exists():
        print("Patch target missing:", path)
        return []
    original = path.read_text(encoding="utf-8")
    updated = original
    applied = []
    for label, old, new in replacements:
        if old in updated:
            updated = updated.replace(old, new)
            applied.append(label)
        elif new in updated:
            print("Patch already present:", label)
        else:
            print("Patch pattern not found:", label, "in", path)
    if updated != original:
        path.write_text(updated, encoding="utf-8")
    return applied

def apply_fast_source_patches():
    patches = []
    patches += patch_text_file(
        FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py",
        [
            (
                "allow_flash_attention_backends",
                "        with sdpa_kernel(SDPBackend.MATH):",
                "        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):",
            ),
            (
                "guard_text2semantic_cuda_cache_clear",
                "                if torch.cuda.is_available():\n                    torch.cuda.empty_cache()",
                "                if os.getenv(\"FISH_CLEAR_CUDA_CACHE_AFTER_REQUEST\", \"0\") == \"1\" and torch.cuda.is_available():\n                    torch.cuda.empty_cache()",
            ),
        ],
    )
    patches += patch_text_file(
        FISH_REPO / "fish_speech" / "inference_engine" / "__init__.py",
        [
            ("import_os_for_cache_guard", "import gc\n", "import gc\nimport os\n"),
            (
                "guard_engine_cuda_cache_clear",
                "        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            gc.collect()",
                "        if os.getenv(\"FISH_CLEAR_CUDA_CACHE_AFTER_REQUEST\", \"0\") == \"1\" and torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            gc.collect()",
            ),
        ],
    )
    for rel in ["tools/server/model_manager.py", "tools/run_webui.py"]:
        patches += patch_text_file(
            FISH_REPO / rel,
            [(f"light_warmup_{rel}", "max_new_tokens=1024,", "max_new_tokens=128,")],
        )
    patches += patch_text_file(
        FISH_REPO / "awesome_webui" / "src" / "App.tsx",
        [("awesome_default_max_tokens", "  maxNewTokens: 2048,", "  maxNewTokens: 384,")],
    )
    return patches

def install_cloudflared():
    if shutil.which("cloudflared"):
        return
    Path("/content/bin").mkdir(parents=True, exist_ok=True)
    run([
        "curl", "-L", "--fail", "--output", "/content/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    run(["chmod", "+x", "/content/bin/cloudflared"])
    os.environ["PATH"] = "/content/bin:" + os.environ["PATH"]

def wait_for_log_regex(log_path, pattern, *, timeout_seconds=60):
    deadline = time.time() + timeout_seconds
    compiled = re.compile(pattern)
    while time.time() < deadline:
        text = read_tail(log_path, lines=120)
        match = compiled.search(text)
        if match:
            return match.group(0)
        time.sleep(1)
    raise RuntimeError(f"Timed out waiting for pattern {pattern!r} in {log_path}. Tail:\n{read_tail(log_path, lines=120)}")

def sanitize_voice_id(value):
    slug = re.sub(r"[^A-Za-z0-9_-]+", "-", value.strip()).strip("-")
    return slug[:80] or "voice"

print("Python:", sys.version)
FISH_NOTEBOOK_BOOTSTRAPPED = True
print("Fish Speech target:", FISH_SPEECH_REPO_URL, FISH_SPEECH_REF)
print("Artifact root:", ARTIFACT_ROOT)
if DRY_RUN:
    print("Dry run enabled: Drive, GPU, model download, servers, and tunnels will be simulated.")


In [ ]:
#@title 2. Clone Fish Speech and apply local fast patches
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

if MOUNT_DRIVE and not DRY_RUN:
    from google.colab import drive
    drive.mount("/content/drive")
elif MOUNT_DRIVE:
    print("Dry run: skipping Drive mount.")
else:
    print("Drive mount disabled. Artifacts stay in ephemeral Colab storage:", ARTIFACT_ROOT)

if DRY_RUN:
    if not FISH_REPO.exists():
        raise RuntimeError(f"Dry run expects an existing fake or local Fish Speech repo at {FISH_REPO}")
    fish_commit = os.environ.get("FISH_DRY_RUN_COMMIT", "dry-run-fish-commit")
    print("Dry run: skipping git clone/fetch/checkout.")
elif FISH_REPO.exists():
    run(["git", "remote", "set-url", "origin", FISH_SPEECH_REPO_URL], cwd=FISH_REPO, check=False)
    run(["git", "fetch", "origin", "--tags"], cwd=FISH_REPO)
else:
    run(["git", "clone", FISH_SPEECH_REPO_URL, str(FISH_REPO)])

if not DRY_RUN and FISH_SPEECH_REF.strip():
    run(["git", "checkout", FISH_SPEECH_REF.strip()], cwd=FISH_REPO)

if not DRY_RUN:
    fish_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=FISH_REPO, text=True).strip()
required_paths = [
    FISH_REPO / "pyproject.toml",
    FISH_REPO / "tools" / "run_webui.py",
    FISH_REPO / "tools" / "api_server.py",
    FISH_REPO / "tools" / "server" / "views.py",
    FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py",
    FISH_REPO / "awesome_webui" / "package.json",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise RuntimeError("Fish Speech checkout is missing expected server/WebUI files: " + json.dumps(missing, indent=2))

if APPLY_FAST_SOURCE_PATCHES:
    source_patches = apply_fast_source_patches()
else:
    source_patches = []

print("Fish Speech repo:", FISH_REPO)
print("Fish Speech commit:", fish_commit)
print("Applied local source patches:", source_patches or "none")
print("API server:", FISH_REPO / "tools" / "api_server.py")
print("Gradio WebUI reloads a separate model:", FISH_REPO / "tools" / "run_webui.py")
print("Awesome WebUI is frontend-only and calls /v1/tts from the API server after npm build.")


In [ ]:
#@title 3. Configure run paths, GPU, secrets, and install dependencies
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("fish_commit")
RUN_ID = str(globals().get("RUN_ID", "")).strip() or os.environ.get("FISHAUDIO_RUN_ID", "").strip() or default_run_id()
RUN_DIR = ARTIFACT_ROOT / "runs" / RUN_ID
LOGS_DIR = RUN_DIR / "logs"
OUTPUTS_DIR = RUN_DIR / "outputs"
MANIFESTS_DIR = RUN_DIR / "manifests"
UPLOADS_DIR = WORK_ROOT / "uploads"
CHECKPOINT_DIR = WORK_ROOT / "checkpoints" / "s2-pro"
CACHE_ROOT = WORK_ROOT / "cache"
for path in [RUN_DIR, LOGS_DIR, OUTPUTS_DIR, MANIFESTS_DIR, UPLOADS_DIR, CHECKPOINT_DIR, CACHE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "HF_HOME": str(CACHE_ROOT / "hf"),
    "HF_HUB_CACHE": str(CACHE_ROOT / "hf" / "hub"),
    "TRANSFORMERS_CACHE": str(CACHE_ROOT / "hf" / "transformers"),
})

HF_TOKEN = read_colab_secret("HF_TOKEN", required=True)
os.environ["HF_TOKEN"] = HF_TOKEN

api_auth_source = "disabled"
if ENABLE_API_AUTH:
    API_AUTH_TOKEN = str(API_AUTH_TOKEN).strip()
    if API_AUTH_TOKEN:
        api_auth_source = "parameter"
    else:
        secret_value = read_colab_secret(API_AUTH_SECRET_NAME, required=False)
        if secret_value:
            API_AUTH_TOKEN = secret_value.strip()
            api_auth_source = f"secret:{API_AUTH_SECRET_NAME}"
        else:
            API_AUTH_TOKEN = secrets.token_urlsafe(24)
            api_auth_source = "generated"
else:
    API_AUTH_TOKEN = ""
os.environ["FISHAUDIO_API_AUTH_TOKEN"] = API_AUTH_TOKEN

gpus = require_minimum_vram(24)
FISH_UV_EXTRA = resolve_fish_uv_extra(FISH_UV_EXTRA_REQUEST)

write_json(MANIFESTS_DIR / "runtime.json", {
    "run_id": RUN_ID,
    "dry_run": DRY_RUN,
    "fish_repo": str(FISH_REPO),
    "fish_commit": fish_commit,
    "source_patches": source_patches,
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "cache_root": str(CACHE_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
    "run_dir": str(RUN_DIR),
    "gpus": gpus,
    "fish_uv_extra": FISH_UV_EXTRA,
    "api_auth_enabled": bool(API_AUTH_TOKEN),
    "api_auth_source": api_auth_source,
})

print("Run dir:", RUN_DIR)
print("Ephemeral checkpoint dir:", CHECKPOINT_DIR)
print("GPU inventory:", gpus)
print("Fish uv extra:", FISH_UV_EXTRA)
print("API auth:", "enabled" if API_AUTH_TOKEN else "disabled", f"({api_auth_source})")
if api_auth_source in {"generated", "parameter"}:
    print("API bearer token:", API_AUTH_TOKEN)
elif API_AUTH_TOKEN:
    print("API bearer token loaded from Colab Secret; not printing it here.")

if DRY_RUN:
    print("Dry run: skipping apt, pip, and uv dependency installation.")
else:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-y", "-qq", "install", "ffmpeg", "portaudio19-dev", "libsox-dev"])
    run([sys.executable, "-m", "pip", "install", "-U", "uv", "huggingface_hub", "requests"])

    print("Installing Fish Speech upstream dependencies with uv. A fresh Colab VM can take several minutes.", flush=True)
    run(["uv", "--color", "always", "sync", "--python", "3.12", "--extra", FISH_UV_EXTRA], cwd=FISH_REPO)


In [ ]:
#@title 4. Download Fish Audio S2 Pro weights to ephemeral /content
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("CHECKPOINT_DIR", "LOGS_DIR", "MANIFESTS_DIR", "HF_TOKEN")
MODEL_ID = "fishaudio/s2-pro"
download_log_path = LOGS_DIR / "hf_download.log"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if DRY_RUN:
    (CHECKPOINT_DIR / "codec.pth").write_text("dry-run codec checkpoint placeholder\n", encoding="utf-8")
    download_log_path.write_text("dry run: skipped hf download\n", encoding="utf-8")
    print("Dry run: created checkpoint placeholder in ephemeral work root:", CHECKPOINT_DIR)
elif not (CHECKPOINT_DIR / "codec.pth").exists():
    if shutil.which("hf") is None:
        raise RuntimeError("`hf` CLI was not found after installing huggingface_hub")

    print("Downloading S2 Pro weights to ephemeral /content.", flush=True)
    print("Download log:", download_log_path, flush=True)
    download_env = fish_env()
    download_env["HF_TOKEN"] = HF_TOKEN
    download_env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    download_env.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
    hf_cmd = [
        "hf", "download", MODEL_ID,
        "--local-dir", str(CHECKPOINT_DIR),
        "--max-workers", str(max(1, int(HF_DOWNLOAD_WORKERS))),
    ]
    shell_cmd = " ".join(shlex.quote(str(part)) for part in hf_cmd)
    shell_cmd = f"set -o pipefail; {shell_cmd} 2>&1 | tee -a {shlex.quote(str(download_log_path))}"
    run(["bash", "-lc", shell_cmd], cwd=FISH_REPO, env=download_env)
else:
    print("Checkpoint already present in ephemeral /content:", CHECKPOINT_DIR)

if not (CHECKPOINT_DIR / "codec.pth").exists():
    raise RuntimeError(f"Download completed but codec.pth is missing under {CHECKPOINT_DIR}")

write_json(MANIFESTS_DIR / "model_download.json", {"model_id": MODEL_ID, "checkpoint_dir": str(CHECKPOINT_DIR), "storage": "ephemeral_content", "dry_run": DRY_RUN})


In [ ]:
#@title 5. Start one long-lived authenticated API server
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("CHECKPOINT_DIR", "LOGS_DIR", "API_AUTH_TOKEN")
api_local_url = f"http://127.0.0.1:{API_PORT}"

if "api_proc" in globals():
    stop_process(api_proc)

api_proc = start_api_server(port=API_PORT, compile_model=COMPILE_MODEL, half=USE_HALF, api_key=API_AUTH_TOKEN or None)
wait_for_http(f"{api_local_url}/v1/health", headers=api_headers(), timeout_seconds=900)
print("API healthy:", f"{api_local_url}/v1/health")
print("TTS endpoint:", f"{api_local_url}/v1/tts")
if API_AUTH_TOKEN:
    print("Use HTTP header: Authorization: Bearer <token from cell 3>")
print("API log tail:")
print(read_tail(api_proc["log_path"], lines=80))


In [ ]:
#@title 6. Minimal no-reference TTS API showcase
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("API_PORT", "OUTPUTS_DIR", "API_AUTH_TOKEN")
from IPython.display import Audio, display

if SHOWCASE_STREAMING and SHOWCASE_FORMAT != "wav":
    raise ValueError("SHOWCASE_STREAMING requires SHOWCASE_FORMAT='wav' because upstream streaming only supports WAV.")
if not 100 <= int(SHOWCASE_CHUNK_LENGTH) <= 1000:
    raise ValueError("SHOWCASE_CHUNK_LENGTH must be between 100 and 1000 for upstream ServeTTSRequest validation.")
if int(SHOWCASE_MAX_NEW_TOKENS) < 1:
    raise ValueError("SHOWCASE_MAX_NEW_TOKENS must be positive for this speed-focused showcase.")

showcase_output = OUTPUTS_DIR / f"showcase_no_reference.{SHOWCASE_FORMAT}"
payload = {
    "text": SHOWCASE_TEXT,
    "references": [],
    "reference_id": None,
    "format": SHOWCASE_FORMAT,
    "latency": "normal",
    "max_new_tokens": int(SHOWCASE_MAX_NEW_TOKENS),
    "chunk_length": int(SHOWCASE_CHUNK_LENGTH),
    "top_p": 0.8,
    "repetition_penalty": 1.1,
    "temperature": 0.8,
    "streaming": bool(SHOWCASE_STREAMING),
    "use_memory_cache": "off",
    "seed": 42,
}
started = time.time()
ttft = None
if DRY_RUN:
    write_silent_wav(showcase_output)
    response_bytes = showcase_output.read_bytes()
    elapsed = time.time() - started
else:
    import requests
    response = requests.post(
        f"http://127.0.0.1:{API_PORT}/v1/tts",
        json=payload,
        headers=api_headers({"Content-Type": "application/json"}),
        timeout=900,
        stream=bool(SHOWCASE_STREAMING),
    )
    if response.status_code != 200:
        error_path = showcase_output.with_suffix(".error.json")
        write_json(error_path, {"status_code": response.status_code, "body": response.text[:4000], "payload": payload})
        raise RuntimeError(f"TTS request failed with HTTP {response.status_code}; see {error_path}")
    if SHOWCASE_STREAMING:
        chunks = []
        for chunk in response.iter_content(chunk_size=65536):
            if chunk:
                if ttft is None:
                    ttft = time.time() - started
                chunks.append(chunk)
        response_bytes = b"".join(chunks)
    else:
        response_bytes = response.content
    elapsed = time.time() - started
    showcase_output.write_bytes(response_bytes)
manifest = {
    "output_path": str(showcase_output),
    "bytes": len(response_bytes),
    "elapsed_seconds": elapsed,
    "ttft_seconds": ttft,
    "reference_id": None,
    "seed": 42,
    "text_chars": len(SHOWCASE_TEXT),
    "max_new_tokens": int(SHOWCASE_MAX_NEW_TOKENS),
    "chunk_length": int(SHOWCASE_CHUNK_LENGTH),
    "dry_run": DRY_RUN,
}
write_json(showcase_output.with_suffix(showcase_output.suffix + ".manifest.json"), manifest)
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("API log tail with token/sec metrics:")
print(read_tail(api_proc["log_path"], lines=120))
display(Audio(filename=str(showcase_output)))

if DOWNLOAD_SHOWCASE_OUTPUT and not DRY_RUN:
    from google.colab import files
    files.download(str(showcase_output))


In [ ]:
#@title 7. Prepare optional reference voice from upload or path
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("FISH_REPO", "RUN_DIR", "MANIFESTS_DIR", "UPLOADS_DIR")

if UPLOAD_REFERENCE_AUDIO and not REFERENCE_AUDIO_PATH.strip() and not DRY_RUN:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        name, data = next(iter(uploaded.items()))
        upload_path = UPLOADS_DIR / Path(name).name
        upload_path.write_bytes(data)
        REFERENCE_AUDIO_PATH = str(upload_path)
        print("Uploaded reference audio:", REFERENCE_AUDIO_PATH)

voice_id = sanitize_voice_id(VOICE_ID)
reference_manifest = None
if REFERENCE_AUDIO_PATH.strip():
    source_audio = Path(REFERENCE_AUDIO_PATH.strip())
    if not DRY_RUN and not source_audio.exists():
        raise FileNotFoundError(f"Reference audio does not exist: {source_audio}")
    if not REFERENCE_TEXT.strip():
        raise ValueError("REFERENCE_TEXT is required for voice cloning.")

    ref_dir = FISH_REPO / "references" / voice_id
    ref_dir.mkdir(parents=True, exist_ok=True)
    wav_path = ref_dir / "sample.wav"
    lab_path = ref_dir / "sample.lab"
    if DRY_RUN:
        write_silent_wav(wav_path, sample_rate=44100)
    else:
        run([
            "ffmpeg", "-hide_banner", "-y",
            "-ss", str(REFERENCE_START_SECONDS),
            "-t", str(REFERENCE_MAX_SECONDS),
            "-i", str(source_audio),
            "-ac", "1", "-ar", "44100", "-vn", str(wav_path),
        ])
    lab_path.write_text(REFERENCE_TEXT.strip() + "\n", encoding="utf-8")

    copied_to_artifacts = False
    if SAVE_REFERENCE_COPY_TO_ARTIFACTS:
        artifact_ref = RUN_DIR / "references" / voice_id
        artifact_ref.mkdir(parents=True, exist_ok=True)
        shutil.copy2(wav_path, artifact_ref / wav_path.name)
        shutil.copy2(lab_path, artifact_ref / lab_path.name)
        copied_to_artifacts = True

    reference_manifest = {
        "voice_id": voice_id,
        "source_audio": str(source_audio),
        "fish_reference_dir": str(ref_dir),
        "fish_reference_audio": str(wav_path),
        "fish_reference_text": str(lab_path),
        "start_seconds": REFERENCE_START_SECONDS,
        "max_seconds": REFERENCE_MAX_SECONDS,
        "sample_rate": 44100,
        "saved_reference_copy_to_artifacts": copied_to_artifacts,
        "dry_run": DRY_RUN,
    }
    write_json(MANIFESTS_DIR / f"reference_{voice_id}.json", reference_manifest)
    print(reference_manifest)
else:
    print("Set REFERENCE_AUDIO_PATH or enable UPLOAD_REFERENCE_AUDIO, and provide REFERENCE_TEXT, to run voice cloning.")


In [ ]:
#@title 8. Optional reference voice TTS through the same API server
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("API_PORT", "OUTPUTS_DIR", "API_AUTH_TOKEN")
from IPython.display import Audio, display
reference_manifest = globals().get("reference_manifest")

if reference_manifest is None:
    print("Skipping reference TTS because no reference voice was prepared.")
else:
    ref_output = OUTPUTS_DIR / f"reference_{voice_id}_smoke.{SHOWCASE_FORMAT}"
    payload = {
        "text": REFERENCE_TTS_TEXT,
        "references": [],
        "reference_id": voice_id,
        "format": SHOWCASE_FORMAT,
        "latency": "normal",
        "max_new_tokens": int(SHOWCASE_MAX_NEW_TOKENS),
        "chunk_length": int(SHOWCASE_CHUNK_LENGTH),
        "top_p": 0.8,
        "repetition_penalty": 1.1,
        "temperature": 0.8,
        "streaming": False,
        "use_memory_cache": "on",
        "seed": 42,
    }
    started = time.time()
    if DRY_RUN:
        write_silent_wav(ref_output)
        response_bytes = ref_output.read_bytes()
        elapsed = time.time() - started
    else:
        import requests
        response = requests.post(
            f"http://127.0.0.1:{API_PORT}/v1/tts",
            json=payload,
            headers=api_headers({"Content-Type": "application/json"}),
            timeout=900,
        )
        elapsed = time.time() - started
        if response.status_code != 200:
            error_path = ref_output.with_suffix(".error.json")
            write_json(error_path, {"status_code": response.status_code, "body": response.text[:4000], "payload": payload, "elapsed_seconds": elapsed})
            raise RuntimeError(f"Reference TTS failed with HTTP {response.status_code}; see {error_path}")
        ref_output.write_bytes(response.content)
        response_bytes = response.content
    manifest = {"output_path": str(ref_output), "bytes": len(response_bytes), "elapsed_seconds": elapsed, "reference_id": voice_id, "seed": 42, "text_chars": len(REFERENCE_TTS_TEXT), "dry_run": DRY_RUN}
    write_json(ref_output.with_suffix(ref_output.suffix + ".manifest.json"), manifest)
    print(manifest)
    print("API log tail with token/sec metrics:")
    print(read_tail(api_proc["log_path"], lines=120))
    display(Audio(filename=str(ref_output)))


In [ ]:
#@title 9. Optional efficient Awesome WebUI on the existing API server
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("API_PORT", "FISH_REPO")
awesome_web_url = None

if BUILD_AWESOME_WEBUI:
    if API_AUTH_TOKEN:
        print("Awesome WebUI is skipped because upstream API bearer auth protects /ui and the browser cannot attach a header to the initial page load.")
        print("For a browser UI, either disable ENABLE_API_AUTH and put Cloudflare Access/Tailscale in front, or use the API-only tunnel below.")
    else:
        if shutil.which("npm") is None:
            raise RuntimeError("npm is required to build Awesome WebUI.")
        run(["npm", "install"], cwd=FISH_REPO / "awesome_webui")
        run(["npm", "run", "build"], cwd=FISH_REPO / "awesome_webui")
        if not (FISH_REPO / "awesome_webui" / "dist" / "index.html").exists():
            raise RuntimeError("Awesome WebUI build did not create awesome_webui/dist/index.html")
        awesome_web_url = f"http://127.0.0.1:{API_PORT}/ui"
        wait_for_http(awesome_web_url, timeout_seconds=120)
        print("Awesome WebUI ready on the existing API server:", awesome_web_url)
else:
    print("Awesome WebUI build skipped. API-only path avoids an extra model load.")

if START_GRADIO_WEBUI:
    print("Starting Gradio will load a second independent model engine. This is intentionally off by default.")
    gradio_proc = start_gradio_webui(compile_model=COMPILE_MODEL, half=USE_HALF)
    wait_for_http("http://127.0.0.1:7860", timeout_seconds=900)
    print("Gradio ready: http://127.0.0.1:7860")


In [ ]:
#@title 10. Optional public API tunnel
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("API_PORT", "LOGS_DIR")
api_local_url = f"http://127.0.0.1:{API_PORT}"
tunnel_proc = None
public_api_base_url = None

if PUBLIC_TUNNEL_MODE == "none":
    print("Public tunnel skipped.")
elif PUBLIC_TUNNEL_MODE == "cloudflare_quick":
    install_cloudflared()
    tunnel_proc = start_process(
        "cloudflared-quick",
        ["cloudflared", "tunnel", "--url", api_local_url, "--no-autoupdate"],
        cwd=WORK_ROOT,
        log_path=LOGS_DIR / "cloudflared_quick.log",
    )
    public_api_base_url = wait_for_log_regex(LOGS_DIR / "cloudflared_quick.log", r"https://[-A-Za-z0-9.]+\.trycloudflare\.com", timeout_seconds=90)
    print("Cloudflare quick tunnel URL:", public_api_base_url)
elif PUBLIC_TUNNEL_MODE == "cloudflare_named":
    install_cloudflared()
    tunnel_token = read_colab_secret(CLOUDFLARED_TUNNEL_TOKEN_SECRET, required=True)
    tunnel_proc = start_process(
        "cloudflared-named",
        ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", tunnel_token],
        cwd=WORK_ROOT,
        log_path=LOGS_DIR / "cloudflared_named.log",
        display_command=["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", "REDACTED"],
    )
    if CLOUDFLARED_CUSTOM_DOMAIN.strip():
        public_api_base_url = CLOUDFLARED_CUSTOM_DOMAIN.strip().rstrip("/")
        print("Configured public API URL:", public_api_base_url)
    else:
        print("Named tunnel started. Set CLOUDFLARED_CUSTOM_DOMAIN if you want this notebook to print the public URL.")
else:
    raise ValueError(f"Unknown PUBLIC_TUNNEL_MODE: {PUBLIC_TUNNEL_MODE}")

if public_api_base_url:
    print("TTS endpoint:", public_api_base_url.rstrip("/") + "/v1/tts")
    print("Curl shape:")
    print("curl -X POST " + shlex.quote(public_api_base_url.rstrip("/") + "/v1/tts") + " \\")
    if API_AUTH_TOKEN:
        print("  -H 'Authorization: Bearer <token from cell 3>' \\")
    print("  -H 'Content-Type: application/json' \\")
    print("  --data '{\"text\":\"hello\",\"format\":\"wav\",\"max_new_tokens\":384,\"chunk_length\":1000}' \\")
    print("  --output out.wav")


In [ ]:
#@title 11. Inspect logs, manifest, and optional artifact download
if "FISH_NOTEBOOK_BOOTSTRAPPED" not in globals():
    raise RuntimeError("Notebook bootstrap state is missing. Run cells 0 and 1 first, then rerun this cell. You do not need to reupload the notebook.")

require_vars("RUN_DIR", "OUTPUTS_DIR", "LOGS_DIR", "MANIFESTS_DIR", "fish_commit")
active_outputs = [str(path) for path in sorted(OUTPUTS_DIR.glob("*")) if path.is_file()]
manifest = {
    "run_id": RUN_ID,
    "dry_run": DRY_RUN,
    "run_dir": str(RUN_DIR),
    "outputs_dir": str(OUTPUTS_DIR),
    "logs_dir": str(LOGS_DIR),
    "fish_repo": str(FISH_REPO),
    "fish_commit": fish_commit,
    "source_patches": source_patches,
    "api_auth_enabled": bool(API_AUTH_TOKEN),
    "public_api_base_url": public_api_base_url,
    "awesome_web_url": awesome_web_url,
    "outputs": active_outputs,
}
write_json(MANIFESTS_DIR / "final_manifest.json", manifest)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

for log_path in sorted(LOGS_DIR.glob("*.log")):
    print("\n====", log_path, "====")
    print(read_tail(log_path, lines=80))

if DOWNLOAD_RUN_ZIP and not DRY_RUN:
    archive_base = RUN_DIR / f"{RUN_ID}_artifacts"
    zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
    from google.colab import files
    files.download(zip_path)
